# 11 — Agentic Tool-Use with SochDB + Gemini Function Calling

This notebook builds a **complete AI agent** that uses **Gemini function calling** with SochDB as its tool backend.

The agent can:
| Tool | Description |
|---|---|
| `search_knowledge` | Search the vector knowledge base (RAG) |
| `check_cache` | Check semantic cache before making expensive calls |
| `store_cache` | Save results to semantic cache |
| `store_fact` | Store a fact in the KV store |
| `get_fact` | Retrieve a fact from the KV store |
| `add_to_graph` | Add an entity/relationship to the knowledge graph |
| `query_graph` | Query relationships in the knowledge graph |

In [27]:
!pip install openai sochdb jupyter

In [28]:
!export GEMINI_API_KEY=os.environ.get("GEMINI_API_KEY", "YOUR_API_KEY_HERE")

In [29]:
import os, json, time, shutil
from openai import OpenAI
from sochdb import Database
from sochdb.namespace import CollectionConfig

# Set the key directly in os.environ
os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY", "YOUR_API_KEY_HERE")

client = OpenAI(
    api_key=os.environ.get("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

EMBED_MODEL = "gemini-embedding-001"
CHAT_MODEL  = "gemini-3.1-flash-lite-preview"
DIM = 3072

def embed(text: str) -> list[float]:
    resp = client.embeddings.create(model=EMBED_MODEL, input=text)
    return resp.data[0].embedding

def embed_batch(texts: list[str]) -> list[list[float]]:
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in resp.data]

print("Setup complete")

Setup complete


## 1. Initialize SochDB and Seed Knowledge Base

In [30]:
DB_PATH = "./agent_tools_db"
if os.path.exists(DB_PATH):
    shutil.rmtree(DB_PATH)

db = Database.open(DB_PATH)
ns = db.create_namespace("agent")

# Knowledge base collection
kb_config = CollectionConfig(name="knowledge_base", dimension=DIM)
kb = ns.create_collection(kb_config)

# Seed with domain knowledge
knowledge = [
    ("kb_1", "Python is a high-level programming language created by Guido van Rossum in 1991. It emphasizes code readability."),
    ("kb_2", "Rust is a systems programming language focused on safety, speed, and concurrency. It was developed by Mozilla."),
    ("kb_3", "SochDB is an embedded database written in Rust. It supports vector search, ACID transactions, and temporal graphs."),
    ("kb_4", "Machine learning models use training data to learn patterns. Common types include supervised, unsupervised, and reinforcement learning."),
    ("kb_5", "Docker containers package applications with their dependencies for consistent deployment across environments."),
    ("kb_6", "Kubernetes orchestrates containerized applications across clusters, handling scaling, load balancing, and self-healing."),
    ("kb_7", "GraphQL is a query language for APIs that allows clients to request exactly the data they need."),
    ("kb_8", "WebAssembly (Wasm) enables running compiled code in web browsers at near-native speed."),
]

texts = [text for _, text in knowledge]
vecs = embed_batch(texts)
for (doc_id, text), vec in zip(knowledge, vecs):
    kb.insert(id=doc_id, vector=vec, metadata={"text": text}, content=text)

# Add some graph nodes
db.add_node("agent", "python", "language", {"name": "Python", "year": 1991})
db.add_node("agent", "rust", "language", {"name": "Rust", "year": 2010})
db.add_node("agent", "sochdb", "database", {"name": "SochDB", "written_in": "Rust"})
db.add_node("agent", "guido", "person", {"name": "Guido van Rossum"})
db.add_edge("agent", "guido", "created", "python", {})
db.add_edge("agent", "sochdb", "written_in", "rust", {})

print(f"Knowledge base: {len(knowledge)} docs")
print(f"Graph: 4 nodes, 2 edges")

Knowledge base: 8 docs
Graph: 4 nodes, 2 edges


## 2. Define Tool Functions (SochDB-Backed)

Each tool function wraps a SochDB operation.

In [31]:
CACHE_NAME = "agent_cache"

def search_knowledge(query: str, k: int = 3) -> str:
    """Search the knowledge base for relevant documents."""
    q_vec = embed(query)
    results = kb.vector_search(vector=q_vec, k=k)
    docs = []
    for r in results:
        meta = r.metadata or {}
        docs.append({"id": r.id, "score": round(r.score, 4), "text": meta.get("text", meta.get("_content", ""))})
    return json.dumps(docs)


def check_cache(query: str) -> str:
    """Check if a similar query has been answered before (semantic cache)."""
    q_vec = embed(query)
    cached = db.cache_get(CACHE_NAME, q_vec, threshold=0.92)
    if cached:
        return json.dumps({"hit": True, "cached_answer": cached})
    return json.dumps({"hit": False})


def store_cache(query: str, answer: str) -> str:
    """Store a query-answer pair in the semantic cache."""
    q_vec = embed(query)
    db.cache_put(CACHE_NAME, query, answer, q_vec)
    return json.dumps({"stored": True})


def store_fact(key: str, value: str) -> str:
    """Store a key-value fact in the database."""
    db.put(f"facts/{key}".encode(), value.encode())
    return json.dumps({"stored": True, "key": key})


def get_fact(key: str) -> str:
    """Retrieve a fact from the database."""
    val = db.get(f"facts/{key}".encode())
    if val:
        return json.dumps({"found": True, "key": key, "value": val.decode()})
    return json.dumps({"found": False, "key": key})


def add_to_graph(entity_id: str, entity_type: str, name: str, 
                 related_to: str = "", relation: str = "") -> str:
    """Add an entity (and optional relationship) to the knowledge graph."""
    db.add_node("agent", entity_id, entity_type, {"name": name})
    if related_to and relation:
        db.add_edge("agent", entity_id, relation, related_to, {})
    return json.dumps({"added": True, "entity": entity_id})


def query_graph(entity_id: str) -> str:
    """Query the knowledge graph for an entity's relationships."""
    neighbors = db.get_neighbors("agent", entity_id)
    return json.dumps({"entity": entity_id, "neighbors": neighbors})


# Map function names to implementations
TOOL_FUNCTIONS = {
    "search_knowledge": search_knowledge,
    "check_cache": check_cache,
    "store_cache": store_cache,
    "store_fact": store_fact,
    "get_fact": get_fact,
    "add_to_graph": add_to_graph,
    "query_graph": query_graph,
}

print(f"Registered {len(TOOL_FUNCTIONS)} tool functions")

Registered 7 tool functions


## 3. Define Tool Schemas for Gemini Function Calling

In [32]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_knowledge",
            "description": "Search the knowledge base for information. Use this when you need factual information to answer a question.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"},
                    "k": {"type": "integer", "description": "Number of results (default 3)"},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_cache",
            "description": "Check if a similar question has been answered before. Always check cache first before searching.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The query to check"},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "store_cache",
            "description": "Store a query-answer pair in the semantic cache for future reuse.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The original question"},
                    "answer": {"type": "string", "description": "The answer to cache"},
                },
                "required": ["query", "answer"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "store_fact",
            "description": "Store a key-value fact for later retrieval.",
            "parameters": {
                "type": "object",
                "properties": {
                    "key": {"type": "string", "description": "Fact key (e.g. 'user_preference')"},
                    "value": {"type": "string", "description": "Fact value"},
                },
                "required": ["key", "value"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_fact",
            "description": "Retrieve a previously stored fact by key.",
            "parameters": {
                "type": "object",
                "properties": {
                    "key": {"type": "string", "description": "Fact key to look up"},
                },
                "required": ["key"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "add_to_graph",
            "description": "Add an entity and optional relationship to the knowledge graph.",
            "parameters": {
                "type": "object",
                "properties": {
                    "entity_id": {"type": "string", "description": "Unique entity ID"},
                    "entity_type": {"type": "string", "description": "Type (e.g. person, language, tool)"},
                    "name": {"type": "string", "description": "Display name"},
                    "related_to": {"type": "string", "description": "Optional: ID of related entity"},
                    "relation": {"type": "string", "description": "Optional: relationship type"},
                },
                "required": ["entity_id", "entity_type", "name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "query_graph",
            "description": "Query the knowledge graph for an entity's relationships and neighbors.",
            "parameters": {
                "type": "object",
                "properties": {
                    "entity_id": {"type": "string", "description": "Entity ID to query"},
                },
                "required": ["entity_id"],
            },
        },
    },
]

print(f"Defined {len(TOOLS)} tool schemas for Gemini")

Defined 7 tool schemas for Gemini


## 4. Agent Loop: Gemini Function Calling

The agent loop:
1. Send user message + tools to Gemini
2. If Gemini returns tool calls → execute them → feed results back
3. Repeat until Gemini produces a final text response

In [33]:
def run_agent(user_message: str, verbose: bool = True) -> str:
    """
    Run the agent loop with Gemini function calling.
    
    The agent can call SochDB-backed tools iteratively until it
    has enough information to answer.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful AI assistant with access to tools. "
                "ALWAYS check the cache first before searching the knowledge base. "
                "After answering a question from the knowledge base, cache the result. "
                "Use the knowledge graph to find relationships between entities. "
                "Store important user-provided facts for later recall."
            ),
        },
        {"role": "user", "content": user_message},
    ]
    
    max_iterations = 8
    
    for iteration in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {iteration + 1} ---")
        
        response = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=messages,
            tools=TOOLS,
            temperature=0.2,
        )
        
        choice = response.choices[0]
        message = choice.message
        
        # If no tool calls, we have the final answer
        if not message.tool_calls:
            if verbose:
                print(f"Final answer (no more tool calls)")
            return message.content
        
        # Append assistant message with tool_calls
        messages.append(message)
        
        # Execute each tool call
        for tool_call in message.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            
            if verbose:
                print(f"  Tool: {fn_name}({json.dumps(fn_args, ensure_ascii=False)})")
            
            # Execute the tool
            fn = TOOL_FUNCTIONS.get(fn_name)
            if fn:
                try:
                    result = fn(**fn_args)
                except Exception as e:
                    result = json.dumps({"error": str(e)})
            else:
                result = json.dumps({"error": f"Unknown tool: {fn_name}"})
            
            if verbose:
                preview = result[:200] + "..." if len(result) > 200 else result
                print(f"  Result: {preview}")
            
            # Append tool result
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })
    
    return "Agent reached maximum iterations without producing a final answer."

print("Agent loop ready")

Agent loop ready


## 5. Run the Agent: Knowledge Base Search

In [34]:
answer = run_agent("What programming language is SochDB written in, and what features does it support?")
print(f"\n{'='*60}")
print(f"ANSWER: {answer}")


--- Iteration 1 ---
  Tool: check_cache({"query": "What programming language is SochDB written in, and what features does it support?"})
  Result: {"hit": false}

--- Iteration 2 ---
  Tool: search_knowledge({"query": "What is SochDB?"})
  Result: [{"id": "kb_3", "score": 0.7774, "text": "SochDB is an embedded database written in Rust. It supports vector search, ACID transactions, and temporal graphs."}, {"id": "kb_5", "score": 0.5261, "text": ...

--- Iteration 3 ---
  Tool: store_cache({"answer": "SochDB is written in Rust. Its key features include vector search, ACID transactions, and support for temporal graphs.", "query": "What programming language is SochDB written in, and what features does it support?"})
  Result: {"stored": true}

--- Iteration 4 ---
Final answer (no more tool calls)

ANSWER: SochDB is written in **Rust**. 

Its key features include:
*   **Vector search**
*   **ACID transactions**
*   **Temporal graphs**


## 6. Run the Agent: Cache Hit on Repeated Query

The agent should find the cached answer from the previous query.

In [35]:
# Similar question — should hit the cache
answer2 = run_agent("Tell me about SochDB and what language it uses")
print(f"\n{'='*60}")
print(f"ANSWER: {answer2}")


--- Iteration 1 ---
  Tool: check_cache({"query": "What is SochDB and what language does it use?"})
  Result: {"hit": true, "cached_answer": "SochDB is written in Rust. Its key features include vector search, ACID transactions, and support for temporal graphs."}

--- Iteration 2 ---
Final answer (no more tool calls)

ANSWER: SochDB is a database system written in **Rust**.

Its key features include:
*   **Vector Search:** Designed to handle high-dimensional vector data, making it suitable for AI and machine learning applications.
*   **ACID Transactions:** Ensures data integrity and reliability through Atomicity, Consistency, Isolation, and Durability.
*   **Temporal Graphs:** Supports the storage and querying of temporal graph data, allowing you to analyze how relationships change over time.


## 7. Run the Agent: Store Facts + Graph Relationships

In [36]:
answer3 = run_agent(
    "Remember that my favorite language is Python and I'm working on a project called DataFlow. "
    "DataFlow uses SochDB for storage. Add these to the knowledge graph and store as facts."
)
print(f"\n{'='*60}")
print(f"ANSWER: {answer3}")


--- Iteration 1 ---
  Tool: store_fact({"key": "favorite_language", "value": "Python"})
  Result: {"stored": true, "key": "favorite_language"}
  Tool: store_fact({"key": "current_project", "value": "DataFlow"})
  Result: {"stored": true, "key": "current_project"}
  Tool: add_to_graph({"entity_type": "language", "name": "Python", "entity_id": "python_lang"})
  Result: {"added": true, "entity": "python_lang"}
  Tool: add_to_graph({"entity_type": "project", "entity_id": "dataflow_proj", "name": "DataFlow", "relation": "uses_language", "related_to": "python_lang"})
  Result: {"added": true, "entity": "dataflow_proj"}
  Tool: add_to_graph({"name": "SochDB", "entity_type": "database", "entity_id": "sochdb_db"})
  Result: {"added": true, "entity": "sochdb_db"}
  Tool: add_to_graph({"name": "DataFlow", "entity_id": "dataflow_proj", "entity_type": "project", "related_to": "sochdb_db", "relation": "uses_storage"})
  Result: {"added": true, "entity": "dataflow_proj"}

--- Iteration 2 ---
Final a

## 8. Run the Agent: Recall Stored Facts + Graph Query

In [37]:
answer4 = run_agent("What's my favorite language? And what project am I working on?")
print(f"\n{'='*60}")
print(f"ANSWER: {answer4}")


--- Iteration 1 ---
  Tool: get_fact({"key": "favorite_language"})
  Result: {"found": true, "key": "favorite_language", "value": "Python"}
  Tool: get_fact({"key": "current_project"})
  Result: {"found": true, "key": "current_project", "value": "DataFlow"}

--- Iteration 2 ---
Final answer (no more tool calls)

ANSWER: Your favorite language is Python, and you are currently working on the DataFlow project.


In [38]:
# Query graph relationships
answer5 = run_agent("What do you know about the relationships of SochDB in the knowledge graph?")
print(f"\n{'='*60}")
print(f"ANSWER: {answer5}")


--- Iteration 1 ---
  Tool: query_graph({"entity_id": "SochDB"})
  Result: {"entity": "SochDB", "neighbors": {"neighbors": []}}

--- Iteration 2 ---
Final answer (no more tool calls)

ANSWER: I have checked the knowledge graph, but I currently do not have any information or relationships stored for "SochDB." 

If you have information about SochDB that you would like me to store or if you would like me to look for information about it elsewhere, please let me know!


## 9. Run the Agent: Multi-Step Reasoning

A complex question that requires multiple tool calls.

In [39]:
import time

def run_agent(user_message: str, verbose: bool = True) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful AI assistant with access to tools. "
                "ALWAYS check the cache first before searching the knowledge base. "
                "After answering a question from the knowledge base, cache the result. "
                "Use the knowledge graph to find relationships between entities. "
                "Store important user-provided facts for later recall."
            ),
        },
        {"role": "user", "content": user_message},
    ]

    max_iterations = 8

    for iteration in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {iteration + 1} ---")

        # Retry logic for Gemini 500 errors
        response = None
        for attempt in range(3):
            try:
                response = client.chat.completions.create(
                    model=CHAT_MODEL,
                    messages=messages,
                    tools=TOOLS,
                    temperature=0.2,
                )
                break  # success
            except Exception as e:
                if "500" in str(e) or "INTERNAL" in str(e):
                    wait = 2 ** attempt
                    print(f"  [Gemini 500] Retrying in {wait}s... (attempt {attempt+1}/3)")
                    time.sleep(wait)
                else:
                    raise  # non-500 error, don't retry

        if response is None:
            return "Agent failed: Gemini returned 500 after 3 retries."

        choice = response.choices[0]
        message = choice.message

        if not message.tool_calls:
            if verbose:
                print(f"Final answer (no more tool calls)")
            return message.content

        messages.append(message)

        for tool_call in message.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)

            if verbose:
                print(f"  Tool: {fn_name}({json.dumps(fn_args, ensure_ascii=False)})")

            fn = TOOL_FUNCTIONS.get(fn_name)
            if fn:
                try:
                    result = fn(**fn_args)
                except Exception as e:
                    result = json.dumps({"error": str(e)})
            else:
                result = json.dumps({"error": f"Unknown tool: {fn_name}"})

            if verbose:
                preview = result[:200] + "..." if len(result) > 200 else result
                print(f"  Result: {preview}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })

    return "Agent reached maximum iterations without producing a final answer."

In [40]:
response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=messages,
    tools=TOOLS,
    temperature=0.2,
    extra_body={"parallel_tool_calls": None}  # strip unsupported Gemini field
)

NameError: name 'messages' is not defined

In [ ]:
answer6 = run_agent(
    "I want to compare Docker and Kubernetes. Search the knowledge base for both, "
    "then summarize the key differences and store the comparison as a fact."
)
print(f"\n{'='*60}")
print(f"ANSWER: {answer6}")

## 10. Cleanup

In [ ]:
db.close()
shutil.rmtree(DB_PATH, ignore_errors=True)
print("Done! Database cleaned up.")

In [ ]:
# vector search, semantic cache, AND knowledge graph — one DB handle
db = Database.open("./mydb")
cached = db.cache_get("cache", query_vec, threshold=0.92)  # semantic cache
results = kb.vector_search(vector=query_vec, k=3)           # RAG
neighbors = db.get_neighbors("app", "sochdb")               # graph